# RNN Lecture 2 — Quick Recall

Before we build RNNs from scratch, let us quickly recall why RNNs are needed.

## 1. Sequential Data
In sequential data, order matters.

Examples:
- Sentence: "I am happy"
- Time-series: sensor readings over time
- Stock prices over days
- Audio signals
- Character sequences

Unlike standard feedforward networks, sequence models must process information in order.

---

## 2. Hidden State Idea
RNN introduces a **hidden state** that acts like a running memory.

At each time step:
- the model reads current input
- combines it with past memory
- updates its new memory

So the hidden state carries information from previous time steps.

---

## 3. Basic RNN Update
At time step $t$:

$$
h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)
$$

where:
- $x_t$ = current input
- $h_{t-1}$ = previous hidden state
- $h_t$ = new hidden state

If output is needed:

$$
y_t = W_{hy} h_t + b_y
$$

This means:
- input flows through time
- memory gets updated repeatedly
- output can be taken at the final step or at every step

---

## 4. Two Common Output Styles

### Many-to-One
Whole sequence $\rightarrow$ one output

Example:
- sentiment classification
- sequence classification

### Many-to-Many
Sequence $\rightarrow$ output at each step

Example:
- next character prediction
- part-of-speech tagging
- sequence labeling

---

## What we do today
Today we will follow this path:

**Practical Implementation $\rightarrow$ Observation $\rightarrow$ Theory $\rightarrow$ Limitation $\rightarrow$ LSTM/GRU intuition**

So first, we build basic RNNs ourselves and see how they behave in practice.

In [1]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

# Small toy example just to recall hidden state evolution
# Sequence length = 4, input dimension = 2, hidden dimension = 3

x_seq = [
    np.array([[1.0], [0.0]]),
    np.array([[0.5], [1.0]]),
    np.array([[0.0], [1.0]]),
    np.array([[1.0], [1.0]])
]

W_xh = np.array([
    [0.5, -0.2],
    [0.3,  0.8],
    [-0.6, 0.1]
])

W_hh = np.array([
    [0.4, 0.1, -0.2],
    [0.0, 0.5,  0.3],
    [0.2, -0.4, 0.6]
])

b_h = np.array([[0.0], [0.0], [0.0]])

h = np.zeros((3, 1))

print("Initial hidden state:\n", h)

for t, x_t in enumerate(x_seq, start=1):
    h = np.tanh(W_xh @ x_t + W_hh @ h + b_h)
    print(f"\nTime step {t}")
    print("Input x_t:\n", x_t)
    print("Hidden state h_t:\n", h)

Initial hidden state:
 [[0.]
 [0.]
 [0.]]

Time step 1
Input x_t:
 [[1.]
 [0.]]
Hidden state h_t:
 [[ 0.462]
 [ 0.291]
 [-0.537]]

Time step 2
Input x_t:
 [[0.5]
 [1. ]]
Hidden state h_t:
 [[ 0.355]
 [ 0.733]
 [-0.498]]

Time step 3
Input x_t:
 [[0.]
 [1.]]
Hidden state h_t:
 [[ 0.114]
 [ 0.769]
 [-0.398]]

Time step 4
Input x_t:
 [[1.]
 [1.]]
Hidden state h_t:
 [[ 0.464]
 [ 0.878]
 [-0.771]]


## Concept Check

1. Why can a normal feedforward neural network struggle with sequence data?

2. In an RNN, what does the hidden state represent?

3. In the equation

$$
h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)
$$

what role does $h_{t-1}$ play?

4. Which setup is many-to-one?
   - (a) Sentiment classification of a sentence
   - (b) Predicting one output for every character in a sequence

5. Which setup is many-to-many?
   - (a) One movie review $\rightarrow$ one sentiment label
   - (b) One input sequence $\rightarrow$ one output at each time step

# Many-to-One RNN — Practical Implementation

Now we implement RNN from scratch for a **Many-to-One** problem.

## Many-to-One Definition

Entire sequence → Single output

Examples:
- Sentiment classification
- Sequence classification
- Time-series classification

Example:

"I love this movie"

Input sequence:

I → love → this → movie

Final Output:

Positive sentiment

---

## RNN Idea

We process sequence step-by-step

At each time step:

$$
h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)
$$

After full sequence:

Final hidden state becomes representation of entire sequence

$$
y = W_{hy} h_T + b_y
$$

So:

Sequence → Hidden states → Final hidden state → Output

---

## Toy Example

We create:

Vocabulary:

- good → positive
- bad → negative

Simple sequences:

- [good, good, good] → positive
- [bad, bad, bad] → negative
- [good, bad, good] → positive

This helps visualize RNN behavior.

In [3]:
import numpy as np

np.random.seed(42)

# -----------------------------
# Vocabulary
# -----------------------------

word_to_vec = {
    "good": np.array([[1.0], [0.0]]),
    "bad":  np.array([[0.0], [1.0]])
}

# -----------------------------
# Toy Dataset
# -----------------------------

dataset = [
    (["good", "good", "good","good"], 1),
    (["bad", "bad", "bad", "bad"], 0),
    (["good", "bad", "good"], 1),
    (["bad", "good", "bad"], 0)
]

# -----------------------------
# Model Parameters
# -----------------------------

input_size = 2
hidden_size = 4
output_size = 1

W_xh = np.random.randn(hidden_size, input_size) * 0.5
W_hh = np.random.randn(hidden_size, hidden_size) * 0.5
W_hy = np.random.randn(output_size, hidden_size) * 0.5

b_h = np.zeros((hidden_size,1))
b_y = np.zeros((output_size,1))

# -----------------------------
# Forward Function
# -----------------------------

def forward(sequence):
    
    h = np.zeros((hidden_size,1))
    
    hidden_states = []
    
    for word in sequence:
        
        x = word_to_vec[word]
        
        h = np.tanh(W_xh @ x + W_hh @ h + b_h)
        
        hidden_states.append(h)
    
    y = W_hy @ h + b_y
    
    return y, hidden_states


# -----------------------------
# Run Forward Pass
# -----------------------------

for sequence, label in dataset:
    
    y, hidden_states = forward(sequence)
    
    print("\nSequence:", sequence)
    print("Hidden States Evolution:")
    
    for i, h in enumerate(hidden_states):
        print(f"t={i+1} -> {h.T}")
    
    print("Final Output:", y)
    print("Actual Label:", label)


Sequence: ['good', 'good', 'good', 'good']
Hidden States Evolution:
t=1 -> [[ 0.243  0.313 -0.117  0.658]]
t=2 -> [[ 0.149 -0.031 -0.539  0.43 ]]
t=3 -> [[ 0.226  0.614 -0.251  0.521]]
t=4 -> [[ 0.29  -0.165 -0.371  0.467]]
Final Output: [[0.213]]
Actual Label: 1

Sequence: ['bad', 'bad', 'bad', 'bad']
Hidden States Evolution:
t=1 -> [[-0.069  0.642 -0.117  0.366]]
t=2 -> [[ 0.063  0.136 -0.185 -0.004]]
t=3 -> [[-0.003  0.664 -0.041  0.389]]
t=4 -> [[ 0.031  0.052 -0.261  0.028]]
Final Output: [[0.15]]
Actual Label: 0

Sequence: ['good', 'bad', 'good']
Hidden States Evolution:
t=1 -> [[ 0.243  0.313 -0.117  0.658]]
t=2 -> [[-0.166  0.386 -0.539  0.054]]
t=3 -> [[0.466 0.367 0.23  0.514]]
Final Output: [[-0.142]]
Actual Label: 1

Sequence: ['bad', 'good', 'bad']
Hidden States Evolution:
t=1 -> [[-0.069  0.642 -0.117  0.366]]
t=2 -> [[ 0.363 -0.292 -0.185  0.382]]
t=3 -> [[-0.273  0.813 -0.487  0.384]]
Final Output: [[0.472]]
Actual Label: 0


# Concept Check

1. In Many-to-One RNN, when is output generated?

2. Why do we use the final hidden state for prediction?

3. If sequence length increases, what happens to hidden state?

4. In sentiment classification, why is Many-to-One suitable?

5. What does hidden state represent after processing full sequence?

# Many-to-Many RNN — Practical Implementation

Now we move to a second important RNN setup:

## Many-to-Many Definition

Input sequence $\rightarrow$ Output at each time step

Examples:
- Next character prediction
- Next token prediction
- Part-of-speech tagging
- Sequence labeling

---

## Core Idea

Instead of waiting until the full sequence ends, we produce an output at every time step.

At each time step:

$$
h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)
$$

and now also

$$
y_t = W_{hy}h_t + b_y
$$

So the flow becomes:

$$
x_1 \rightarrow h_1 \rightarrow y_1
$$

$$
x_2 \rightarrow h_2 \rightarrow y_2
$$

$$
x_3 \rightarrow h_3 \rightarrow y_3
$$

and so on.

---

## Practical Toy Example

We use a simple next-character style task.

Vocabulary:

- A
- B
- C

Input sequence:

A, B, C

Target idea:

At each step, predict the next symbol pattern.

This is not yet a full language model.
The goal is to understand:

- output at each time step
- hidden state carries memory forward
- prediction changes as sequence evolves

---

## What students should observe

1. Hidden state keeps evolving through time  
2. Output is generated at every time step  
3. Later outputs depend on earlier sequence information  
4. This setup is more expressive than many-to-one for sequence generation and labeling

In [4]:
import numpy as np

np.random.seed(7)
np.set_printoptions(precision=3, suppress=True)

# -----------------------------
# Vocabulary
# -----------------------------
char_to_vec = {
    "A": np.array([[1.0], [0.0], [0.0]]),
    "B": np.array([[0.0], [1.0], [0.0]]),
    "C": np.array([[0.0], [0.0], [1.0]])
}

idx_to_char = {0: "A", 1: "B", 2: "C"}

# -----------------------------
# Toy Sequence
# -----------------------------
sequence = ["A", "B", "C", "A"]

# -----------------------------
# Model Dimensions
# -----------------------------
input_size = 3
hidden_size = 4
output_size = 3

# -----------------------------
# Parameters
# -----------------------------
W_xh = np.random.randn(hidden_size, input_size) * 0.4
W_hh = np.random.randn(hidden_size, hidden_size) * 0.4
W_hy = np.random.randn(output_size, hidden_size) * 0.4

b_h = np.zeros((hidden_size, 1))
b_y = np.zeros((output_size, 1))

# -----------------------------
# Softmax Function
# -----------------------------
def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)

# -----------------------------
# Forward Pass (Many-to-Many)
# -----------------------------
h = np.zeros((hidden_size, 1))

hidden_states = []
outputs = []

for t, token in enumerate(sequence, start=1):
    x_t = char_to_vec[token]
    
    h = np.tanh(W_xh @ x_t + W_hh @ h + b_h)
    y_t = W_hy @ h + b_y
    p_t = softmax(y_t)
    
    hidden_states.append(h)
    outputs.append(p_t)
    
    predicted_idx = np.argmax(p_t)
    predicted_char = idx_to_char[predicted_idx]
    
    print(f"\nTime step {t}")
    print("Input token:", token)
    print("Hidden state:", h.T)
    print("Output probabilities:", p_t.T)
    print("Predicted token:", predicted_char)


Time step 1
Input token: A
Hidden state: [[ 0.589  0.162 -0.     0.236]]
Output probabilities: [[0.486 0.232 0.282]]
Predicted token: A

Time step 2
Input token: B
Hidden state: [[-0.218 -0.31  -0.111 -0.516]]
Output probabilities: [[0.259 0.442 0.299]]
Predicted token: B

Time step 3
Input token: C
Hidden state: [[ 0.302  0.235 -0.156  0.541]]
Output probabilities: [[0.462 0.272 0.265]]
Predicted token: A

Time step 4
Input token: A
Hidden state: [[ 0.391 -0.105  0.589 -0.355]]
Output probabilities: [[0.331 0.195 0.474]]
Predicted token: C


## Concept Check

1. In many-to-many RNN, when is output generated?

2. What is the main difference between many-to-one and many-to-many?

3. Why can many-to-many RNN be used for next-token prediction?

4. Does the hidden state at time $t$ depend only on $x_t$?

5. Why might later outputs be different even if the current input token repeats?

# Training RNN From Scratch

So far:

- We built RNN
- We generated outputs
- But predictions were random

Why?

Because weights were random and **no learning happened**.

---

## Training Components

To train RNN we need:

1. Loss Function  
2. Forward Pass  
3. Backward Pass  
4. Weight Update  

---

## Training Flow

For each sequence:

Step 1 — Forward pass through time

$$
h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1})
$$

Step 2 — Final Prediction

$$
y = W_{hy}h_T
$$

Step 3 — Compute Loss

$$
L = (y - y_{true})^2
$$

Step 4 — Update Weights

Gradient Descent:

$$
W = W - \eta \nabla W
$$

---

## What Students Should Observe

- Loss reduces gradually
- Predictions improve
- RNN starts learning sequence patterns

Now we implement this.

In [5]:
import numpy as np

np.random.seed(1)

# -------------------------
# Vocabulary
# -------------------------

word_to_vec = {
    "good": np.array([[1.0],[0.0]]),
    "bad":  np.array([[0.0],[1.0]])
}

# -------------------------
# Dataset
# -------------------------

dataset = [
    (["good","good","good"],1),
    (["bad","bad","bad"],0),
    (["good","bad","good"],1),
    (["bad","good","bad"],0)
]

# -------------------------
# Model Dimensions
# -------------------------

input_size = 2
hidden_size = 4
output_size = 1

# -------------------------
# Initialize Weights
# -------------------------

W_xh = np.random.randn(hidden_size,input_size)*0.5
W_hh = np.random.randn(hidden_size,hidden_size)*0.5
W_hy = np.random.randn(output_size,hidden_size)*0.5

b_h = np.zeros((hidden_size,1))
b_y = np.zeros((output_size,1))

learning_rate = 0.01

# -------------------------
# Training Loop
# -------------------------

epochs = 50

for epoch in range(epochs):

    total_loss = 0
    
    for sequence, label in dataset:
        
        # Forward pass
        
        h = np.zeros((hidden_size,1))
        states = []
        
        for word in sequence:
            
            x = word_to_vec[word]
            
            h = np.tanh(W_xh @ x + W_hh @ h + b_h)
            states.append(h)
        
        y = W_hy @ h + b_y
        
        # Loss
        
        loss = (y - label)**2
        total_loss += loss
        
        # Simple Gradient Approximation
        
        dy = 2*(y-label)
        
        dW_hy = dy @ states[-1].T
        
        # Backprop through last state (simplified)
        
        dh = W_hy.T @ dy
        
        dtanh = (1 - states[-1]**2) * dh
        
        dW_xh = dtanh @ word_to_vec[sequence[-1]].T
        
        # Update
        
        W_hy -= learning_rate * dW_hy
        W_xh -= learning_rate * dW_xh
    
    print(f"Epoch {epoch+1}, Loss: {total_loss.flatten()[0]:.4f}")

Epoch 1, Loss: 1.9237
Epoch 2, Loss: 1.6286
Epoch 3, Loss: 1.3821
Epoch 4, Loss: 1.1757
Epoch 5, Loss: 1.0027
Epoch 6, Loss: 0.8574
Epoch 7, Loss: 0.7351
Epoch 8, Loss: 0.6321
Epoch 9, Loss: 0.5451
Epoch 10, Loss: 0.4715
Epoch 11, Loss: 0.4091
Epoch 12, Loss: 0.3561
Epoch 13, Loss: 0.3110
Epoch 14, Loss: 0.2725
Epoch 15, Loss: 0.2395
Epoch 16, Loss: 0.2112
Epoch 17, Loss: 0.1869
Epoch 18, Loss: 0.1658
Epoch 19, Loss: 0.1476
Epoch 20, Loss: 0.1318
Epoch 21, Loss: 0.1180
Epoch 22, Loss: 0.1059
Epoch 23, Loss: 0.0954
Epoch 24, Loss: 0.0861
Epoch 25, Loss: 0.0779
Epoch 26, Loss: 0.0707
Epoch 27, Loss: 0.0644
Epoch 28, Loss: 0.0587
Epoch 29, Loss: 0.0537
Epoch 30, Loss: 0.0493
Epoch 31, Loss: 0.0453
Epoch 32, Loss: 0.0418
Epoch 33, Loss: 0.0387
Epoch 34, Loss: 0.0359
Epoch 35, Loss: 0.0334
Epoch 36, Loss: 0.0312
Epoch 37, Loss: 0.0292
Epoch 38, Loss: 0.0274
Epoch 39, Loss: 0.0258
Epoch 40, Loss: 0.0244
Epoch 41, Loss: 0.0232
Epoch 42, Loss: 0.0220
Epoch 43, Loss: 0.0210
Epoch 44, Loss: 0.02

# Concept Check

1. Why do we compute loss?

2. What does decreasing loss indicate?

3. Why do we update weights?

4. Why is training done over multiple epochs?

5. What is learning rate controlling?

# Training Real RNN Using PyTorch

So far we:

- Implemented RNN from scratch
- Observed hidden state updates
- Implemented simple training

Now we implement **Real RNN using PyTorch**

---

## Why Use nn.RNN?

PyTorch provides:

- Efficient RNN implementation
- Built-in backpropagation
- GPU support
- Stable training

Instead of manually writing:

- hidden loops
- gradients
- updates

We define:

$$
\text{model} = nn.RNN(...)
$$

---

## Many-to-One Task

We solve:

Sentiment Classification

Sequence → Final Output

Example:

["good", "good", "bad"] → Positive

---

## Training Flow

1. Convert words to vectors  
2. Pass through RNN  
3. Take final hidden state  
4. Pass through linear layer  
5. Compute loss  
6. Backprop  
7. Update weights

---

Now we implement real RNN.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

# -------------------------
# Vocabulary
# -------------------------

word_to_idx = {
    "good":0,
    "bad":1
}

# -------------------------
# Dataset
# -------------------------

dataset = [
    (["good","good","good"],1),
    (["bad","bad","bad"],0),
    (["good","bad","good"],1),
    (["bad","good","bad"],0)
]

# -------------------------
# Convert to tensor
# -------------------------

def encode(sequence):
    return torch.tensor([word_to_idx[w] for w in sequence])

X = [encode(seq) for seq,_ in dataset]
y = torch.tensor([label for _,label in dataset]).float().unsqueeze(1)

# -------------------------
# Model
# -------------------------

class SimpleRNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.embedding = nn.Embedding(2,4)
        self.rnn = nn.RNN(4,8,batch_first=True)
        self.fc = nn.Linear(8,1)
        
    def forward(self,x):
        
        x = self.embedding(x)
        
        output, hidden = self.rnn(x)
        
        final_hidden = hidden.squeeze(0)
        
        y = self.fc(final_hidden)
        
        return y


model = SimpleRNN()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# -------------------------
# Training
# -------------------------

epochs = 100

for epoch in range(epochs):
    
    total_loss = 0
    
    for i in range(len(X)):
        
        optimizer.zero_grad()
        
        output = model(X[i].unsqueeze(0))
        
        loss = criterion(output, y[i].unsqueeze(0))
        
        loss.backward()
        
        optimizer.step()
        
        total_loss += loss.item()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss {total_loss:.4f}")

Epoch 0, Loss 2.2086
Epoch 10, Loss 0.0036
Epoch 20, Loss 0.0011
Epoch 30, Loss 0.0004
Epoch 40, Loss 0.0002
Epoch 50, Loss 0.0001
Epoch 60, Loss 0.0000
Epoch 70, Loss 0.0000
Epoch 80, Loss 0.0000
Epoch 90, Loss 0.0000
